# Extra Trees Classifier Experiment - Classical Baseline

**Project:** How Does Graph Construction Influence the Effectiveness of Weisfeiler-Lehman Graph Kernels for Fine-Grained Plant-Stress Classification from Leaf Images?

**Task:** 57-class OLID-I plant stress classification  
**Model:** Extremely Randomized Trees (`ExtraTreesClassifier`)  
**Notebook:** `New_Models_Baselines/02_Extra_Trees/02_Extra_Trees_Experiment.ipynb`  
**Provenance:** 5-fold stratified cross-validation on 4,387 accepted samples using identical fold splits and features as classical baselines (NB-18) and Logistic Regression.

---
## Experimental Protocol & Methodological Principles

| Dimension | Specification | Rationale |
|-----------|---------------|-----------|
| **Dataset** | 4,387 accepted OLID-I images (`quality_flag == 'ACCEPT'`) | Identical dataset partition across all research experiments |
| **Features** | 38 tabular numerical features (morphological, color, GLCM texture) | Features derived before fold merge to prevent partition leakage |
| **Target** | 57 fine-grained plant-stress classes | Standard project multi-class benchmark |
| **Cross-Validation** | 5 stratified folds from `cv_fold_assignments.csv` | Preserves exact fold membership for paired comparison |
| **Estimator** | `sklearn.ensemble.ExtraTreesClassifier` | Ensemble of randomized cut-point decision trees |
| **Hyperparameters** | `n_estimators=300`, `class_weight='balanced'`, `random_state=42`, `n_jobs=-1` | Documented standard baseline without post-hoc tuning |
| **Scaling Policy** | **No feature scaling applied** | Decision trees and ensemble splits are strictly invariant to monotonic scaling |
| **Timing Policy** | `time.perf_counter()` with warm-up | Distinguishes amortized batch throughput from dedicated single-sample latency |
| **Evaluation Metrics** | Macro-F1 (primary), Accuracy, Weighted-F1 | Sample standard deviation (`ddof=1`) from raw fold scores |


In [1]:
import os
import sys
import time
import json
import warnings as _warnings_module
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # non-interactive backend for headless rendering
import matplotlib.pyplot as plt
import sklearn
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score, f1_score

sw_versions = {
    "python":       sys.version.split()[0],
    "scikit-learn": sklearn.__version__,
    "numpy":        np.__version__,
    "pandas":       pd.__version__,
}
print("Software versions:")
for k, v in sw_versions.items():
    print("  {:20s}: {}".format(k, v))


Software versions:
  python              : 3.11.9
  scikit-learn        : 1.5.0
  numpy               : 1.26.4
  pandas              : 2.3.3


In [2]:
# ── Configuration & Paths ──────────────────────────────────────────────────
# Resolve workspace root dynamically whether executed from root or notebook folder
cwd = os.getcwd()
if os.path.basename(cwd) == "02_Extra_Trees":
    WORKSPACE = os.path.dirname(os.path.dirname(cwd))
elif os.path.basename(cwd) == "New_Models_Baselines":
    WORKSPACE = os.path.dirname(cwd)
elif os.path.isdir(os.path.join(cwd, "New_Models_Baselines")):
    WORKSPACE = cwd
else:
    p = cwd
    WORKSPACE = cwd
    while p and os.path.dirname(p) != p:
        if os.path.isdir(os.path.join(p, "New_Models_Baselines")):
            WORKSPACE = p
            break
        p = os.path.dirname(p)

DATA_DIR  = os.path.join(WORKSPACE, "full_pipeline")
OUT_DIR   = os.path.join(WORKSPACE, "New_Models_Baselines", "results")
FIG_DIR   = os.path.join(WORKSPACE, "New_Models_Baselines", "figures")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

FEAT_CSV = os.path.join(DATA_DIR, "olid_full_features.csv")
FOLD_CSV = os.path.join(DATA_DIR, "cv_fold_assignments.csv")
BL_SUM   = os.path.join(WORKSPACE, "full_pipeline", "results", "classical_baselines",
                         "classical_baselines_summary.csv")
WL_SUM   = os.path.join(WORKSPACE, "full_pipeline", "results", "wl_svm_classification",
                         "wl_svm_summary.csv")
LR_SUM   = os.path.join(OUT_DIR, "logistic_regression_summary.csv")

# ── Experiment Hyperparameters ─────────────────────────────────────────────
RANDOM_STATE          = 42   # fixed seed for reproducibility
N_FOLDS               = 5    # five-fold stratified CV (pre-assigned splits)
N_TIMING_REPS         = 20   # repeated batch prediction calls for amortized latency
N_SINGLE_TIMING_REPS  = 50   # repeated single-sample calls for dedicated benchmark

# ── Extra Trees Model Hyperparameters ──────────────────────────────────────
# ExtraTreesClassifier draws random cut-points for each candidate feature split,
# significantly reducing variance compared to standard random forests.
ET_PARAMS = {
    "n_estimators": 300,        # 300 randomized trees for stable ensemble aggregation
    "criterion":    "gini",     # Gini impurity for split quality
    "max_depth":    None,       # unconstrained depth; trees expand until pure
    "min_samples_split": 2,     # minimum samples required to split internal node
    "min_samples_leaf":  1,     # minimum samples required at leaf node
    "max_features": "sqrt",     # sqrt(38) ~ 6 features considered per random split
    "bootstrap":    False,      # standard Extra Trees uses the entire training fold per tree
    "class_weight": "balanced", # balances unequal class frequencies
    "random_state": RANDOM_STATE,
    "n_jobs":       -1,         # utilize all CPU cores
}

print("Configuration loaded:")
print("  WORKSPACE            :", WORKSPACE)
print("  RANDOM_STATE         :", RANDOM_STATE)
print("  N_FOLDS              :", N_FOLDS)
print("  N_TIMING_REPS        :", N_TIMING_REPS, "(amortized batch prediction)")
print("  N_SINGLE_TIMING_REPS :", N_SINGLE_TIMING_REPS, "(dedicated single-sample benchmark)")
print("  ET_PARAMS            :", ET_PARAMS)
print("  OUT_DIR              :", OUT_DIR)
print("  FIG_DIR              :", FIG_DIR)


Configuration loaded:
  WORKSPACE            : C:\Users\merug\OneDrive\Desktop\FINAL_YEAR_PROJECT\WL_Graph_Representation
  RANDOM_STATE         : 42
  N_FOLDS              : 5
  N_TIMING_REPS        : 20 (amortized batch prediction)
  N_SINGLE_TIMING_REPS : 50 (dedicated single-sample benchmark)
  ET_PARAMS            : {'n_estimators': 300, 'criterion': 'gini', 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'bootstrap': False, 'class_weight': 'balanced', 'random_state': 42, 'n_jobs': -1}
  OUT_DIR              : C:\Users\merug\OneDrive\Desktop\FINAL_YEAR_PROJECT\WL_Graph_Representation\New_Models_Baselines\results
  FIG_DIR              : C:\Users\merug\OneDrive\Desktop\FINAL_YEAR_PROJECT\WL_Graph_Representation\New_Models_Baselines\figures


In [3]:
# ════════════════════════════════════════════════════════════════════════════
# SECTION 2 - DATA VALIDATION (9-point checklist with status accumulator)
# ════════════════════════════════════════════════════════════════════════════

assert os.path.isfile(FEAT_CSV), "Feature CSV not found: " + FEAT_CSV
assert os.path.isfile(FOLD_CSV), "Fold CSV not found: " + FOLD_CSV

df_full = pd.read_csv(FEAT_CSV)
df_acc  = df_full[df_full["quality_flag"] == "ACCEPT"].copy().reset_index(drop=True)

META_COLS = [
    "image_path", "image_name", "class", "crop", "condition",
    "quality_flag", "mask_area_ratio", "failure_reason"
]
# Crucial: derive FEATURE_COLS BEFORE the fold merge to prevent 'fold' entering features
FEATURE_COLS = [c for c in df_acc.columns if c not in META_COLS]

df_folds = pd.read_csv(FOLD_CSV)
df_acc   = df_acc.merge(df_folds[["image_path", "fold"]], on="image_path", how="left")

val_checks = []

# Check 1: Row count
ok1 = len(df_acc) == 4387
val_checks.append((1, "Row count", "PASS" if ok1 else "FAIL", "{} rows (expected 4387)".format(len(df_acc))))
assert ok1, "Expected 4387 rows, got " + str(len(df_acc))

# Check 2: Feature columns
ok2 = len(FEATURE_COLS) == 38
val_checks.append((2, "Feature columns", "PASS" if ok2 else "FAIL", "{} features (expected 38)".format(len(FEATURE_COLS))))
assert ok2, "Expected 38 feature columns, got " + str(len(FEATURE_COLS))

# Check 3: Target classes
n_classes = df_acc["class"].nunique()
ok3 = n_classes == 57
val_checks.append((3, "Target classes", "PASS" if ok3 else "FAIL", "{} unique classes (expected 57)".format(n_classes)))
assert ok3, "Expected 57 target classes, got " + str(n_classes)

# Check 4: NaN in features
nan_total = df_acc[FEATURE_COLS].isna().sum().sum()
ok4 = nan_total == 0
val_checks.append((4, "NaN in features", "PASS" if ok4 else "FAIL", "{} NaNs detected".format(nan_total)))
assert ok4, "Found NaN values in feature matrix"

# Check 5: Inf in features
inf_total = int(np.isinf(df_acc[FEATURE_COLS].values).sum())
ok5 = inf_total == 0
val_checks.append((5, "Inf in features", "PASS" if ok5 else "FAIL", "{} Infs detected".format(inf_total)))
assert ok5, "Found Inf values in feature matrix"

# Check 6: Unique images
n_unique = df_acc["image_path"].nunique()
ok6 = n_unique == len(df_acc)
val_checks.append((6, "Unique images", "PASS" if ok6 else "FAIL", "{} unique of {} rows".format(n_unique, len(df_acc))))
assert ok6, "Duplicate image paths detected in dataset"

# Check 7: Missing fold assignments
missing_folds = int(df_acc["fold"].isna().sum())
ok7 = missing_folds == 0
val_checks.append((7, "Missing folds", "PASS" if ok7 else "FAIL", "{} missing assignments".format(missing_folds)))
assert ok7, "Unassigned folds detected"

# Check 8: All 5 folds present with valid sizes
fold_counts = df_acc["fold"].value_counts().sort_index()
ok8 = set(fold_counts.index.astype(int)) == {0, 1, 2, 3, 4}
val_checks.append((8, "Fold partition sizes", "PASS" if ok8 else "FAIL", str(fold_counts.to_dict())))
assert ok8, "Expected folds 0..4"

# Check 9: Stratification (KL divergence per fold vs overall dist)
overall_dist = df_acc["class"].value_counts(normalize=True)
max_kl = 0.0
for fid in range(N_FOLDS):
    fd = (df_acc[df_acc["fold"] == fid]["class"]
          .value_counts(normalize=True)
          .reindex(overall_dist.index, fill_value=0.0))
    kl = float((overall_dist * np.log((overall_dist + 1e-9) / (fd + 1e-9))).sum())
    max_kl = max(max_kl, kl)
strat_ok = max_kl < 0.05
val_checks.append((9, "Stratification (KL)", "PASS" if strat_ok else "WARN",
                   "Max KL={:.6f} ({})".format(max_kl, "well-stratified" if strat_ok else "potential imbalance")))

# Print checklist
print("Data Validation Checklist:")
for cid, cname, cstatus, cdetail in val_checks:
    print("  [{:4s}] Check {} - {:24s}: {}".format(cstatus, cid, cname, cdetail))

# Determine overall validation status explicitly
n_fail = sum(1 for c in val_checks if c[2] == "FAIL")
n_warn = sum(1 for c in val_checks if c[2] == "WARN")
n_pass = sum(1 for c in val_checks if c[2] == "PASS")

print()
if n_fail > 0:
    print("DATA VALIDATION STATUS: FAILED ({} failed, {} warnings, {} passed)".format(n_fail, n_warn, n_pass))
    raise RuntimeError("Data validation failed with {} errors.".format(n_fail))
elif n_warn > 0:
    print("DATA VALIDATION STATUS: PASSED WITH WARNINGS ({} warning(s), {} passed clean)".format(n_warn, n_pass))
else:
    print("DATA VALIDATION STATUS: ALL CHECKS PASSED ({}/{} checks verified clean)".format(n_pass, len(val_checks)))


Data Validation Checklist:
  [PASS] Check 1 - Row count               : 4387 rows (expected 4387)
  [PASS] Check 2 - Feature columns         : 38 features (expected 38)
  [PASS] Check 3 - Target classes          : 57 unique classes (expected 57)
  [PASS] Check 4 - NaN in features         : 0 NaNs detected
  [PASS] Check 5 - Inf in features         : 0 Infs detected
  [PASS] Check 6 - Unique images           : 4387 unique of 4387 rows
  [PASS] Check 7 - Missing folds           : 0 missing assignments
  [PASS] Check 8 - Fold partition sizes    : {0: 878, 1: 878, 2: 877, 3: 877, 4: 877}
  [PASS] Check 9 - Stratification (KL)     : Max KL=0.000631 (well-stratified)

DATA VALIDATION STATUS: ALL CHECKS PASSED (9/9 checks verified clean)


In [4]:
# ════════════════════════════════════════════════════════════════════════════
# SECTIONS 3 & 4 - EXTRA TREES 5-FOLD CROSS-VALIDATION
# Fresh ExtraTreesClassifier instantiated per fold.
# Raw unscaled features used (tree ensembles invariant to monotonic scaling).
# ════════════════════════════════════════════════════════════════════════════

fold_results  = []
fitted_models = {}

print("Starting 5-fold cross-validation...")
print("  Model: ExtraTreesClassifier(n_estimators=300, class_weight='balanced')")
print("  Preprocessing: None (tree-based models operate directly on raw features).")
print("  Timing metric: Amortized batch prediction time per sample ({} reps, median+IQR)".format(N_TIMING_REPS))
print()

for fid in range(N_FOLDS):
    # ── Split ────────────────────────────────────────────────────────────
    tr_idx = df_acc[df_acc["fold"] != fid].index
    va_idx = df_acc[df_acc["fold"] == fid].index

    X_tr = df_acc.loc[tr_idx, FEATURE_COLS].values
    X_va = df_acc.loc[va_idx, FEATURE_COLS].values
    y_tr = df_acc.loc[tr_idx, "class"].values
    y_va = df_acc.loc[va_idx, "class"].values
    n_val = len(y_va)

    # ── Fresh estimator per fold ─────────────────────────────────────────
    model = ExtraTreesClassifier(**ET_PARAMS)

    # ── Training ─────────────────────────────────────────────────────────
    t_train_start = time.perf_counter()
    model.fit(X_tr, y_tr)
    t_train_end   = time.perf_counter()
    t_train       = t_train_end - t_train_start
    fitted_models[fid] = (model, X_va, y_va)

    # ── Inference timing: Amortized batch prediction time per sample ─────
    # Warm-up call (primes CPU prefetchers and cache)
    _ = model.predict(X_va[:1])

    # Single batch prediction for evaluation metrics
    tb_start = time.perf_counter()
    y_pred   = model.predict(X_va)
    tb_end   = time.perf_counter()
    t_batch  = tb_end - tb_start

    # Repeated full-validation-batch calls divided by n_val
    reps_batch = []
    for _ in range(N_TIMING_REPS):
        r0 = time.perf_counter()
        model.predict(X_va)
        r1 = time.perf_counter()
        reps_batch.append(r1 - r0)
    reps_batch = np.array(reps_batch)
    med_batch_s = float(np.median(reps_batch))
    iqr_batch_s = float(np.percentile(reps_batch, 75) - np.percentile(reps_batch, 25))
    
    # Amortized batch prediction time per sample (ms)
    amortized_batch_ms     = med_batch_s / n_val * 1000.0
    amortized_batch_iqr_ms = iqr_batch_s / n_val * 1000.0

    # ── Metrics ──────────────────────────────────────────────────────────
    acc = accuracy_score(y_va, y_pred)
    mf1 = f1_score(y_va, y_pred, average="macro",    zero_division=0)
    wf1 = f1_score(y_va, y_pred, average="weighted", zero_division=0)

    fold_results.append({
        "model":                             "Extra Trees",
        "fold":                              fid,
        "accuracy":                          acc,
        "macro_f1":                          mf1,
        "weighted_f1":                       wf1,
        "train_time_sec":                    t_train,
        "prediction_batch_time_sec":         t_batch,
        "amortized_batch_ms_per_sample":     amortized_batch_ms,
        "amortized_batch_iqr_ms":            amortized_batch_iqr_ms,
        "n_val_samples":                     n_val,
    })

    print("  fold {}: acc={:.4f} mf1={:.4f} wf1={:.4f} "
          "train={:.3f}s batch={:.5f}s amortized={:.6f}ms/s iqr={:.6f}".format(
              fid, acc, mf1, wf1, t_train, t_batch, amortized_batch_ms, amortized_batch_iqr_ms))

print()
print("All 5 folds trained and evaluated successfully.")


Starting 5-fold cross-validation...
  Model: ExtraTreesClassifier(n_estimators=300, class_weight='balanced')
  Preprocessing: None (tree-based models operate directly on raw features).
  Timing metric: Amortized batch prediction time per sample (20 reps, median+IQR)



  fold 0: acc=0.6207 mf1=0.5864 wf1=0.5984 train=0.475s batch=0.05850s amortized=0.053824ms/s iqr=0.000597


  fold 1: acc=0.5957 mf1=0.5654 wf1=0.5835 train=0.458s batch=0.04746s amortized=0.053873ms/s iqr=0.000849


  fold 2: acc=0.5724 mf1=0.5563 wf1=0.5519 train=0.461s batch=0.05896s amortized=0.066954ms/s iqr=0.002145


  fold 3: acc=0.6032 mf1=0.5780 wf1=0.5830 train=0.521s batch=0.05931s amortized=0.067801ms/s iqr=0.001576


  fold 4: acc=0.5770 mf1=0.5641 wf1=0.5606 train=0.650s batch=0.10598s amortized=0.066480ms/s iqr=0.001906

All 5 folds trained and evaluated successfully.


In [5]:
# ════════════════════════════════════════════════════════════════════════════
# SECTION 4B - DEDICATED SINGLE-SAMPLE PREDICTION BENCHMARK
# Measures isolated, unbatched single-sample query latency after warm-up.
# Contrasted with amortized batch prediction time per sample.
# ════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("DEDICATED SINGLE-SAMPLE PREDICTION BENCHMARK")
print("=" * 70)
print("Protocol:")
print("  - Isolated prediction of single samples (shape: 1 x 38)")
print("  - 10 warm-up prediction calls (discarded) to prime CPU caches")
print("  - {} timed repetitions per fold using time.perf_counter()".format(N_SINGLE_TIMING_REPS))
print()

single_benchmarks = []
for fid in range(N_FOLDS):
    model, X_va, _ = fitted_models[fid]
    x_single = X_va[0:1]  # single test sample
    
    # Warm-up calls
    for _ in range(10):
        _ = model.predict(x_single)
        
    # Timed isolated single-sample predictions
    single_reps = []
    for _ in range(N_SINGLE_TIMING_REPS):
        t0 = time.perf_counter()
        _ = model.predict(x_single)
        t1 = time.perf_counter()
        single_reps.append(t1 - t0)
    single_reps = np.array(single_reps) * 1000.0  # convert to ms
    
    med_single = float(np.median(single_reps))
    iqr_single = float(np.percentile(single_reps, 75) - np.percentile(single_reps, 25))
    single_benchmarks.append({
        "fold": fid,
        "single_sample_median_ms": med_single,
        "single_sample_iqr_ms": iqr_single
    })
    print("  fold {}: single-sample median = {:.4f} ms, IQR = {:.4f} ms".format(
        fid, med_single, iqr_single))

df_single = pd.DataFrame(single_benchmarks)
mean_single_ms  = float(df_single["single_sample_median_ms"].mean())
mean_single_iqr = float(df_single["single_sample_iqr_ms"].mean())

# Attach single-sample metrics to fold_results
for i in range(len(fold_results)):
    fold_results[i]["single_sample_median_ms"] = single_benchmarks[i]["single_sample_median_ms"]
    fold_results[i]["single_sample_iqr_ms"]    = single_benchmarks[i]["single_sample_iqr_ms"]

mean_amort_ms = float(pd.DataFrame(fold_results)["amortized_batch_ms_per_sample"].mean())

print()
print("Inference Timing Comparison (Model-Only, 38 Raw Features, 300 Trees):")
print("  1. Dedicated Single-Sample Prediction : {:.4f} ms (median over isolated calls)".format(mean_single_ms))
print("     Single-Sample IQR                  : {:.4f} ms".format(mean_single_iqr))
print("  2. Amortized Batch Prediction / Sample: {:.6f} ms (batch time divided by n_val)".format(mean_amort_ms))
print("     Ratio (Single / Amortized)         : ~{:.1f}x slower per query".format(mean_single_ms / mean_amort_ms))
print()
print("Architectural Rationale:")
print("  - In batch mode, 300 trees evaluate all ~878 samples using optimized multi-threaded C loops.")
print("  - For an isolated single-sample call, Python function dispatch and per-tree leaf traversal")
print("    incur fixed overhead across 300 separate trees without batch amortization.")
print("  - Neither metric includes leaf image segmentation, graph construction, or feature extraction.")
print("=" * 70)


DEDICATED SINGLE-SAMPLE PREDICTION BENCHMARK
Protocol:
  - Isolated prediction of single samples (shape: 1 x 38)
  - 10 warm-up prediction calls (discarded) to prime CPU caches
  - 50 timed repetitions per fold using time.perf_counter()



  fold 0: single-sample median = 27.9344 ms, IQR = 9.1390 ms


  fold 1: single-sample median = 40.0263 ms, IQR = 26.6124 ms


  fold 2: single-sample median = 27.8422 ms, IQR = 12.8817 ms


  fold 3: single-sample median = 37.6635 ms, IQR = 15.8194 ms


  fold 4: single-sample median = 26.6462 ms, IQR = 1.6289 ms

Inference Timing Comparison (Model-Only, 38 Raw Features, 300 Trees):
  1. Dedicated Single-Sample Prediction : 32.0225 ms (median over isolated calls)
     Single-Sample IQR                  : 13.2163 ms
  2. Amortized Batch Prediction / Sample: 0.061787 ms (batch time divided by n_val)
     Ratio (Single / Amortized)         : ~518.3x slower per query

Architectural Rationale:
  - In batch mode, 300 trees evaluate all ~878 samples using optimized multi-threaded C loops.
  - For an isolated single-sample call, Python function dispatch and per-tree leaf traversal
    incur fixed overhead across 300 separate trees without batch amortization.
  - Neither metric includes leaf image segmentation, graph construction, or feature extraction.


In [6]:
# ════════════════════════════════════════════════════════════════════════════
# SECTION 5 - SUMMARY STATISTICS
# Standard deviation uses ddof=1 (sample std).
# Computed from raw fold-level values, not from rounded figures.
# ════════════════════════════════════════════════════════════════════════════

df_fold = pd.DataFrame(fold_results)

ma  = df_fold["accuracy"].mean()
sa  = df_fold["accuracy"].std(ddof=1)
mm  = df_fold["macro_f1"].mean()
sm  = df_fold["macro_f1"].std(ddof=1)
mw  = df_fold["weighted_f1"].mean()
sw2 = df_fold["weighted_f1"].std(ddof=1)
mt  = df_fold["train_time_sec"].mean()
mb  = df_fold["prediction_batch_time_sec"].mean()
ml  = df_fold["amortized_batch_ms_per_sample"].mean()
mi  = df_fold["amortized_batch_iqr_ms"].mean()
mss = df_fold["single_sample_median_ms"].mean()
msi = df_fold["single_sample_iqr_ms"].mean()

print("Fold-level results (Extra Trees):")
print(df_fold[["fold", "accuracy", "macro_f1", "weighted_f1",
               "train_time_sec", "amortized_batch_ms_per_sample",
               "single_sample_median_ms"]].to_string(index=False))
print()
print("=" * 70)
print("Mean +/- SD (ddof=1, computed from raw fold values):")
print("  Accuracy    : {:.6f} +/- {:.6f}".format(ma, sa))
print("  Macro-F1    : {:.6f} +/- {:.6f}  <-- primary metric".format(mm, sm))
print("  Weighted-F1 : {:.6f} +/- {:.6f}".format(mw, sw2))
print()
print("Timing Breakdown (model-only; features pre-extracted):")
print("  Mean train time                     : {:.4f} s".format(mt))
print("  Mean batch pred time (full fold)    : {:.6f} s".format(mb))
print("  Mean amortized batch pred per sample: {:.6f} ms/sample ({} reps per fold)".format(ml, N_TIMING_REPS))
print("  Mean amortized batch IQR per sample : {:.6f} ms/sample".format(mi))
print("  Mean single-sample pred latency     : {:.4f} ms/sample ({} reps per fold)".format(mss, N_SINGLE_TIMING_REPS))
print("  Mean single-sample pred IQR         : {:.4f} ms/sample".format(msi))
print()
print("CAUTION: Inference timings are model-only (features already extracted).")
print("They are NOT comparable with WL end-to-end latencies.")
print("=" * 70)


Fold-level results (Extra Trees):
 fold  accuracy  macro_f1  weighted_f1  train_time_sec  amortized_batch_ms_per_sample  single_sample_median_ms
    0  0.620729  0.586424     0.598444        0.474747                       0.053824                 27.93440
    1  0.595672  0.565395     0.583487        0.458366                       0.053873                 40.02635
    2  0.572406  0.556315     0.551884        0.461036                       0.066954                 27.84220
    3  0.603193  0.577985     0.583014        0.520971                       0.067801                 37.66350
    4  0.576967  0.564107     0.560593        0.649894                       0.066480                 26.64620

Mean +/- SD (ddof=1, computed from raw fold values):
  Accuracy    : 0.593793 +/- 0.019735
  Macro-F1    : 0.570045 +/- 0.012007  <-- primary metric
  Weighted-F1 : 0.575484 +/- 0.018885

Timing Breakdown (model-only; features pre-extracted):
  Mean train time                     : 0.5130 s
  Mean 

In [7]:
# ════════════════════════════════════════════════════════════════════════════
# SECTION 6 - SAVE RESULTS & CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════

fold_csv_path    = os.path.join(OUT_DIR, "extra_trees_fold_results.csv")
summary_csv_path = os.path.join(OUT_DIR, "extra_trees_summary.csv")
config_json_path = os.path.join(OUT_DIR, "extra_trees_config.json")

df_fold.to_csv(fold_csv_path, index=False)

summary_row = {
    "model":                                  "Extra Trees",
    "mean_accuracy":                          ma,
    "std_accuracy_ddof1":                     sa,
    "mean_macro_f1":                          mm,
    "std_macro_f1_ddof1":                     sm,
    "mean_weighted_f1":                       mw,
    "std_weighted_f1_ddof1":                  sw2,
    "mean_train_time_sec":                    mt,
    "mean_prediction_batch_time_sec":         mb,
    "mean_amortized_batch_ms_per_sample":     ml,
    "mean_amortized_batch_iqr_ms":            mi,
    "mean_single_sample_median_ms":           mss,
    "mean_single_sample_iqr_ms":              msi,
    "n_timing_repetitions_batch":             N_TIMING_REPS,
    "n_timing_repetitions_single":            N_SINGLE_TIMING_REPS,
    "timing_scope":                           "model-only (features pre-extracted; no image processing)",
    "std_ddof":                               1,
}
pd.DataFrame([summary_row]).to_csv(summary_csv_path, index=False)

config = {
    "experiment":   "Extra Trees Classifier - 57-class OLID-I plant stress baseline",
    "data_source":  FEAT_CSV,
    "fold_source":  FOLD_CSV,
    "feature_cols": FEATURE_COLS,
    "target_col":   "class",
    "n_accepted":   int(len(df_acc)),
    "n_features":   int(len(FEATURE_COLS)),
    "n_classes":    57,
    "n_folds":      N_FOLDS,
    "model_params": ET_PARAMS,
    "preprocessing": "None (raw unscaled features; tree-based models invariant to scaling)",
    "timing": {
        "method":                      "time.perf_counter()",
        "train_scope":                 "model.fit() only",
        "inference_scope":             "model-only (no image/graph processing)",
        "warm_up":                     "Warm-up prediction calls discarded before timing",
        "amortized_batch_prediction": {
            "n_reps":                  N_TIMING_REPS,
            "metric":                  "Amortized batch prediction time per sample (median & IQR)",
            "description":             "Repeated validation batch predictions divided by validation size"
        },
        "single_sample_benchmark": {
            "n_reps":                  N_SINGLE_TIMING_REPS,
            "metric":                  "Dedicated isolated single-sample prediction latency (median & IQR)",
            "description":             "Repeated prediction of single unbatched sample (1 x 38) after warm-up"
        },
        "scope_caution":               "Model-only latencies; not comparable to end-to-end WL pipeline."
    },
    "std_ddof":    1,
    "random_seed": RANDOM_STATE,
    "sw_versions": sw_versions,
}
with open(config_json_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("Saved fold results :", fold_csv_path)
print("Saved summary      :", summary_csv_path)
print("Saved config JSON  :", config_json_path)


Saved fold results : C:\Users\merug\OneDrive\Desktop\FINAL_YEAR_PROJECT\WL_Graph_Representation\New_Models_Baselines\results\extra_trees_fold_results.csv
Saved summary      : C:\Users\merug\OneDrive\Desktop\FINAL_YEAR_PROJECT\WL_Graph_Representation\New_Models_Baselines\results\extra_trees_summary.csv
Saved config JSON  : C:\Users\merug\OneDrive\Desktop\FINAL_YEAR_PROJECT\WL_Graph_Representation\New_Models_Baselines\results\extra_trees_config.json


In [8]:
# ════════════════════════════════════════════════════════════════════════════
# SECTION 7 - VERIFIED HISTORICAL BASELINES & PROVENANCE
# Loads verified historical results from NB-18, NB-14, and Logistic Regression.
# ════════════════════════════════════════════════════════════════════════════

df_bls = pd.read_csv(BL_SUM)   # NB-18 classical_baselines_summary.csv
df_wls = pd.read_csv(WL_SUM)   # NB-14 wl_svm_summary.csv
df_lrs = pd.read_csv(LR_SUM)   # Logistic Regression summary

def get_bl_row(df, method_col, method_val):
    row = df[df[method_col] == method_val]
    if len(row) == 0:
        return None, None, None, None, None, None
    acc = float(row["mean_accuracy"].iloc[0])
    acc_sd = float(row["std_accuracy"].iloc[0]) if "std_accuracy" in row else None
    mf1 = float(row["mean_macro_f1"].iloc[0])
    mf1_sd = float(row["std_macro_f1"].iloc[0]) if "std_macro_f1" in row else None
    wf1 = float(row["mean_weighted_f1"].iloc[0]) if "mean_weighted_f1" in row else None
    wf1_sd = float(row["std_weighted_f1"].iloc[0]) if "std_weighted_f1" in row else None
    return acc, acc_sd, mf1, mf1_sd, wf1, wf1_sd

rf_a, rf_as, rf_m, rf_ms, rf_w, rf_ws       = get_bl_row(df_bls, "method", "Random Forest")
lsvm_a, lsvm_as, lsvm_m, lsvm_ms, lsvm_w, _  = get_bl_row(df_bls, "method", "Linear SVM")
rbf_a, rbf_as, rbf_m, rbf_ms, rbf_w, _      = get_bl_row(df_bls, "method", "RBF SVM (Ref)")
knn_a, knn_as, knn_m, knn_ms, knn_w, _      = get_bl_row(df_bls, "method", "KNN (k=5)")

wl2 = df_wls[(df_wls["task"] == "57-class") & (df_wls["graph_type"] == "knn") & (df_wls["wl_height"] == 2)]
wl_a  = float(wl2["mean_accuracy"].iloc[0])
wl_as = float(wl2["std_accuracy"].iloc[0])
wl_m  = float(wl2["mean_macro_f1"].iloc[0])
wl_ms = float(wl2["std_macro_f1"].iloc[0])
wl_w  = float(wl2["mean_weighted_f1"].iloc[0])
wl_ws = float(wl2["std_weighted_f1"].iloc[0])

lr_a  = float(df_lrs["mean_accuracy"].iloc[0])
lr_as = float(df_lrs["std_accuracy_ddof1"].iloc[0])
lr_m  = float(df_lrs["mean_macro_f1"].iloc[0])
lr_ms = float(df_lrs["std_macro_f1_ddof1"].iloc[0])
lr_w  = float(df_lrs["mean_weighted_f1"].iloc[0])
lr_ws = float(df_lrs["std_weighted_f1_ddof1"].iloc[0])

comparison_records = [
    {"Method": "Extra Trees (NEW)",            "Source": "This experiment (5-fold CV)",                      "Accuracy": ma,     "Acc_SD": sa,     "Macro-F1": mm,     "Macro-F1_SD": sm,     "Weighted-F1": mw,     "Weighted-F1_SD": sw2},
    {"Method": "Random Forest",                "Source": "NB-18 verified (classical_baselines_summary.csv)", "Accuracy": rf_a,   "Acc_SD": rf_as,  "Macro-F1": rf_m,   "Macro-F1_SD": rf_ms,  "Weighted-F1": rf_w,   "Weighted-F1_SD": rf_ws},
    {"Method": "Linear SVM",                   "Source": "NB-18 verified (classical_baselines_summary.csv)", "Accuracy": lsvm_a, "Acc_SD": lsvm_as,"Macro-F1": lsvm_m, "Macro-F1_SD": lsvm_ms, "Weighted-F1": lsvm_w, "Weighted-F1_SD": None},
    {"Method": "RBF SVM",                      "Source": "NB-18 verified (classical_baselines_summary.csv)", "Accuracy": rbf_a,  "Acc_SD": rbf_as, "Macro-F1": rbf_m,  "Macro-F1_SD": rbf_ms, "Weighted-F1": rbf_w,  "Weighted-F1_SD": None},
    {"Method": "Logistic Regression",          "Source": "New_Models_Baselines (logistic_regression_summary.csv)", "Accuracy": lr_a, "Acc_SD": lr_as, "Macro-F1": lr_m, "Macro-F1_SD": lr_ms, "Weighted-F1": lr_w, "Weighted-F1_SD": lr_ws},
    {"Method": "k-NN (k=5)",                   "Source": "NB-18 verified (classical_baselines_summary.csv)", "Accuracy": knn_a,  "Acc_SD": knn_as, "Macro-F1": knn_m,  "Macro-F1_SD": knn_ms, "Weighted-F1": knn_w,  "Weighted-F1_SD": None},
    {"Method": "WL+SVM kNN-graph h=2",         "Source": "NB-14 verified (wl_svm_summary.csv)",              "Accuracy": wl_a,   "Acc_SD": wl_as,  "Macro-F1": wl_m,   "Macro-F1_SD": wl_ms,  "Weighted-F1": wl_w,   "Weighted-F1_SD": wl_ws},
]

df_cmp = pd.DataFrame(comparison_records)
cmp_path = os.path.join(OUT_DIR, "extra_trees_comparison.csv")
df_cmp.to_csv(cmp_path, index=False)

print("Comprehensive Comparison Table (Historical Provenance vs. Newly Measured):")
print(df_cmp[["Method", "Source", "Accuracy", "Macro-F1", "Weighted-F1"]].to_string(index=False))
print()
print("Saved comparison table to:", cmp_path)


Comprehensive Comparison Table (Historical Provenance vs. Newly Measured):
              Method                                                 Source  Accuracy  Macro-F1  Weighted-F1
   Extra Trees (NEW)                            This experiment (5-fold CV)  0.593793  0.570045     0.575484
       Random Forest       NB-18 verified (classical_baselines_summary.csv)  0.585360  0.549914     0.565309
          Linear SVM       NB-18 verified (classical_baselines_summary.csv)  0.485526  0.471995     0.489358
             RBF SVM       NB-18 verified (classical_baselines_summary.csv)  0.455438  0.430721     0.453504
 Logistic Regression New_Models_Baselines (logistic_regression_summary.csv)  0.445183  0.420640     0.446560
          k-NN (k=5)       NB-18 verified (classical_baselines_summary.csv)  0.487575  0.410237     0.472101
WL+SVM kNN-graph h=2                    NB-14 verified (wl_svm_summary.csv)  0.424891  0.391110     0.413430

Saved comparison table to: C:\Users\merug\OneDrive\D

In [9]:
# ════════════════════════════════════════════════════════════════════════════
# SECTION 8 - VISUALIZATIONS
# ════════════════════════════════════════════════════════════════════════════

plt.rcParams.update({
    "font.size":      11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "figure.dpi":     150,
})

folds_x   = list(range(N_FOLDS))
mf1_vals  = df_fold["macro_f1"].tolist()
acc_vals  = df_fold["accuracy"].tolist()
wf1_vals  = df_fold["weighted_f1"].tolist()
tr_vals   = df_fold["train_time_sec"].tolist()
batch_lat = df_fold["amortized_batch_ms_per_sample"].tolist()
batch_iqr = df_fold["amortized_batch_iqr_ms"].tolist()

def save_fig(fig, name):
    p = os.path.join(FIG_DIR, name)
    fig.savefig(p, bbox_inches="tight")
    plt.close(fig)
    print("  Saved figure:", p)
    return p

# ── Figure 1: Fold-level Macro-F1 ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(folds_x, mf1_vals, marker="o", lw=2, color="#059669", label="Macro-F1 per fold")
ax.axhline(mm, color="#DC2626", ls="--", lw=1.5, label="Mean = {:.4f}".format(mm))
ax.fill_between(folds_x, [mm - sm]*N_FOLDS, [mm + sm]*N_FOLDS,
                alpha=0.15, color="#DC2626", label="+/-1 SD ({:.4f})".format(sm))
ax.set_xlabel("Fold"); ax.set_ylabel("Macro-F1")
ax.set_title("Extra Trees - Macro-F1 per Fold\n(57-class OLID-I plant stress; n_estimators=300)")
ax.set_xticks(folds_x); ax.set_ylim(0.4, 0.7)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
save_fig(fig, "et_macro_f1_folds.png")

# ── Figure 2: Fold-level Accuracy ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(folds_x, acc_vals, marker="s", lw=2, color="#2563EB", label="Accuracy per fold")
ax.axhline(ma, color="#D97706", ls="--", lw=1.5, label="Mean = {:.4f}".format(ma))
ax.fill_between(folds_x, [ma - sa]*N_FOLDS, [ma + sa]*N_FOLDS,
                alpha=0.15, color="#D97706", label="+/-1 SD ({:.4f})".format(sa))
ax.set_xlabel("Fold"); ax.set_ylabel("Accuracy")
ax.set_title("Extra Trees - Accuracy per Fold\n(57-class OLID-I plant stress; n_estimators=300)")
ax.set_xticks(folds_x); ax.set_ylim(0.4, 0.7)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
save_fig(fig, "et_accuracy_folds.png")

# ── Figure 3: Extra Trees Training Time ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(folds_x, tr_vals, color="#7C3AED", alpha=0.85)
ax.axhline(mt, color="#DC2626", ls="--", lw=1.5, label="Mean = {:.3f}s".format(mt))
for bar, t in zip(bars, tr_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            "{:.2f}s".format(t), ha="center", va="bottom", fontsize=9)
ax.set_xlabel("Fold"); ax.set_ylabel("Training Time (seconds)")
ax.set_title("Extra Trees - Training Time per Fold (n_estimators=300, n_jobs=-1)")
ax.set_xticks(folds_x); ax.set_ylim(0, max(tr_vals) * 1.25)
ax.legend(fontsize=9); ax.grid(True, axis="y", alpha=0.3)
save_fig(fig, "et_training_time.png")

# ── Figure 4: Extra Trees Prediction Time (Amortized Batch vs Dedicated Single)
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(folds_x, batch_lat, color="#0891B2", alpha=0.85, label="Amortized batch time (median)")
ax.errorbar(folds_x, batch_lat, yerr=batch_iqr, fmt="none",
            color="black", capsize=5, lw=1.5, label="IQR")
ax.set_xlabel("Fold"); ax.set_ylabel("Amortized Batch Time (ms / sample)")
ax.set_title("Extra Trees - Amortized Batch Prediction Time per Fold\n"
             "(Repeated validation batch predictions divided by n_val; model-only)")
ax.set_xticks(folds_x); ax.legend(fontsize=9); ax.grid(True, axis="y", alpha=0.3)
save_fig(fig, "et_prediction_time.png")

# ── Figure 5: Macro-F1 Comparison across All Models ─────────────────────────
df_sorted_mf1 = df_cmp.sort_values("Macro-F1", ascending=True)
methods_m   = df_sorted_mf1["Method"].tolist()
mf1_vals_c  = df_sorted_mf1["Macro-F1"].tolist()
mf1_sds_c   = [s if pd.notna(s) else 0.0 for s in df_sorted_mf1["Macro-F1_SD"]]
colors_m    = ["#059669" if "Extra Trees" in m else ("#DC2626" if "Logistic" in m else "#4B5563") for m in methods_m]

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(methods_m, mf1_vals_c, xerr=mf1_sds_c, color=colors_m, alpha=0.85, capsize=4)
for bar, val in zip(bars, mf1_vals_c):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            "{:.4f}".format(val), va="center", ha="left", fontsize=9, fontweight="bold")
ax.set_xlabel("Macro-F1 (Primary Metric)"); ax.set_xlim(0, 0.72)
ax.set_title("Macro-F1 Comparison - Extra Trees vs. Historical Baselines & WL\n"
             "(57-class OLID-I; error bars show +/-1 SD where available)")
ax.axvline(mm, color="#059669", ls=":", alpha=0.6, label="Extra Trees Mean = {:.4f}".format(mm))
ax.legend(fontsize=9, loc="lower right"); ax.grid(True, axis="x", alpha=0.3)
save_fig(fig, "et_comparison_macro_f1.png")

# ── Figure 6: Accuracy Comparison across All Models ─────────────────────────
df_sorted_acc = df_cmp.sort_values("Accuracy", ascending=True)
methods_a   = df_sorted_acc["Method"].tolist()
acc_vals_c  = df_sorted_acc["Accuracy"].tolist()
acc_sds_c   = [s if pd.notna(s) else 0.0 for s in df_sorted_acc["Acc_SD"]]
colors_a    = ["#2563EB" if "Extra Trees" in m else ("#DC2626" if "Logistic" in m else "#4B5563") for m in methods_a]

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(methods_a, acc_vals_c, xerr=acc_sds_c, color=colors_a, alpha=0.85, capsize=4)
for bar, val in zip(bars, acc_vals_c):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            "{:.4f}".format(val), va="center", ha="left", fontsize=9, fontweight="bold")
ax.set_xlabel("Accuracy"); ax.set_xlim(0, 0.72)
ax.set_title("Accuracy Comparison - Extra Trees vs. Historical Baselines & WL\n"
             "(57-class OLID-I; error bars show +/-1 SD where available)")
ax.axvline(ma, color="#2563EB", ls=":", alpha=0.6, label="Extra Trees Mean = {:.4f}".format(ma))
ax.legend(fontsize=9, loc="lower right"); ax.grid(True, axis="x", alpha=0.3)
save_fig(fig, "et_comparison_accuracy.png")

# ── Figure 7: Weighted-F1 Comparison across All Models ──────────────────────
df_sorted_wf1 = df_cmp.sort_values("Weighted-F1", ascending=True)
methods_w   = df_sorted_wf1["Method"].tolist()
wf1_vals_c  = df_sorted_wf1["Weighted-F1"].tolist()
wf1_sds_c   = [s if pd.notna(s) else 0.0 for s in df_sorted_wf1["Weighted-F1_SD"]]
colors_w    = ["#D97706" if "Extra Trees" in m else ("#DC2626" if "Logistic" in m else "#4B5563") for m in methods_w]

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(methods_w, wf1_vals_c, xerr=wf1_sds_c, color=colors_w, alpha=0.85, capsize=4)
for bar, val in zip(bars, wf1_vals_c):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            "{:.4f}".format(val), va="center", ha="left", fontsize=9, fontweight="bold")
ax.set_xlabel("Weighted-F1"); ax.set_xlim(0, 0.72)
ax.set_title("Weighted-F1 Comparison - Extra Trees vs. Historical Baselines & WL\n"
             "(57-class OLID-I; error bars show +/-1 SD where available)")
ax.axvline(mw, color="#D97706", ls=":", alpha=0.6, label="Extra Trees Mean = {:.4f}".format(mw))
ax.legend(fontsize=9, loc="lower right"); ax.grid(True, axis="x", alpha=0.3)
save_fig(fig, "et_comparison_weighted_f1.png")


  Saved figure: C:\Users\merug\OneDrive\Desktop\FINAL_YEAR_PROJECT\WL_Graph_Representation\New_Models_Baselines\figures\et_macro_f1_folds.png
  Saved figure: C:\Users\merug\OneDrive\Desktop\FINAL_YEAR_PROJECT\WL_Graph_Representation\New_Models_Baselines\figures\et_accuracy_folds.png


  Saved figure: C:\Users\merug\OneDrive\Desktop\FINAL_YEAR_PROJECT\WL_Graph_Representation\New_Models_Baselines\figures\et_training_time.png
  Saved figure: C:\Users\merug\OneDrive\Desktop\FINAL_YEAR_PROJECT\WL_Graph_Representation\New_Models_Baselines\figures\et_prediction_time.png


  Saved figure: C:\Users\merug\OneDrive\Desktop\FINAL_YEAR_PROJECT\WL_Graph_Representation\New_Models_Baselines\figures\et_comparison_macro_f1.png
  Saved figure: C:\Users\merug\OneDrive\Desktop\FINAL_YEAR_PROJECT\WL_Graph_Representation\New_Models_Baselines\figures\et_comparison_accuracy.png


  Saved figure: C:\Users\merug\OneDrive\Desktop\FINAL_YEAR_PROJECT\WL_Graph_Representation\New_Models_Baselines\figures\et_comparison_weighted_f1.png


'C:\\Users\\merug\\OneDrive\\Desktop\\FINAL_YEAR_PROJECT\\WL_Graph_Representation\\New_Models_Baselines\\figures\\et_comparison_weighted_f1.png'

In [10]:
# ════════════════════════════════════════════════════════════════════════════
# SECTION 9 - SCIENTIFIC INTERPRETATION
# Observations reported strictly separately from unverified hypotheses.
# ════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("SCIENTIFIC INTERPRETATION - EMPIRICAL FINDINGS & HYPOTHESES")
print("=" * 70)
print()
print("1. EVALUATION PERFORMANCE SUMMARY (57-class OLID-I):")
print("  - Macro-F1    : {:.4f} +/- {:.4f} (ddof=1)  [Primary Metric]".format(mm, sm))
print("  - Accuracy    : {:.4f} +/- {:.4f}".format(ma, sa))
print("  - Weighted-F1 : {:.4f} +/- {:.4f}".format(mw, sw2))
print()
print("2. COMPARATIVE RANKING (Macro-F1, Descending):")
for idx, r in enumerate(df_cmp.sort_values("Macro-F1", ascending=False).to_dict(orient="records"), 1):
    marker = "  <-- THIS EXPERIMENT (NEW)" if "Extra Trees" in r["Method"] else ""
    print("  {:d}. {:30s} : {:.4f}  {}".format(idx, r["Method"], r["Macro-F1"], marker))
print()
print("3. EMPIRICAL OBSERVATIONS vs. HYPOTHESES:")
print()
print("  Observation A (Performance Relative to Historical Baselines & WL):")
print("    - Extra Trees achieves Macro-F1 = {:.4f} +/- {:.4f}, ranking #1 among all evaluated models.".format(mm, sm))
print("    - It exceeds Random Forest ({:.4f}) by {:+.4f} Macro-F1,".format(rf_m, mm - rf_m))
print("      Linear SVM ({:.4f}) by {:+.4f} Macro-F1,".format(lsvm_m, mm - lsvm_m))
print("      Logistic Regression ({:.4f}) by {:+.4f} Macro-F1, and".format(lr_m, mm - lr_m))
print("      the standard WL graph kernel baseline ({:.4f}) by {:+.4f} Macro-F1.".format(wl_m, mm - wl_m))
print()
print("  Observation B & Hypothesis regarding the Advantage of Extra Trees over Random Forest:")
print("    - Observation: Extra Trees improves upon standard Random Forest by +0.0201 Macro-F1 and +0.0084 Accuracy.")
print("    - Hypothesis / Discussion: In Extra Trees, cut-points for candidate split features are chosen purely")
print("      at random rather than optimized via greedy search. In noisy, high-dimensional tabular spaces like")
print("      leaf morphology and texture, extreme randomization frequently reduces ensemble variance without")
print("      substantially increasing bias, explaining its performance edge over standard Random Forest.")
print()
print("  Observation C (Cross-Fold Stability):")
print("    - Macro-F1 across folds spans [{:.4f}, {:.4f}], yielding a standard deviation of {:.4f}.".format(
    min(mf1_vals), max(mf1_vals), sm))
print("    - This standard deviation (0.0120) is less than half that observed for Logistic Regression (0.0298),")
print("      indicating high stability across data partitions.")
print()
print("4. TIMING TERMINOLOGY & BENCHMARK DISTINCTION:")
print("  - Training Time                  : {:.4f} s (300 trees, 38 features, 4387 samples, n_jobs=-1)".format(mt))
print("  - Amortized Batch Prediction Time: {:.6f} ms/sample (IQR: {:.6f} ms)".format(ml, mi))
print("    (Repeated full-validation-batch predictions divided by validation size)")
print("  - Dedicated Single-Sample Latency: {:.4f} ms/sample (IQR: {:.4f} ms)".format(mss, msi))
print("    (Repeated isolated single-sample predictions after warm-up)")
print("  - Amortized batch prediction is ~{:.1f}x faster per sample due to parallel C evaluation across batch samples,".format(mss / ml))
print("    whereas single-sample prediction requires traversing 300 separate tree topologies with individual call overhead.")
print("  - Scope reminder: Both measurements are model-only (pre-extracted features) and cannot")
print("    be compared against end-to-end WL pipeline runtimes.")
print()
print("5. METHODOLOGICAL INTEGRITY:")
print("  - Zero data leakage: fold splits identical to all research experiments.")
print("  - Feature standardization appropriately omitted (trees invariant to monotonic feature scaling).")
print("  - All predictive metrics computed from raw unrounded fold evaluations with ddof=1.")
print("=" * 70)


SCIENTIFIC INTERPRETATION - EMPIRICAL FINDINGS & HYPOTHESES

1. EVALUATION PERFORMANCE SUMMARY (57-class OLID-I):
  - Macro-F1    : 0.5700 +/- 0.0120 (ddof=1)  [Primary Metric]
  - Accuracy    : 0.5938 +/- 0.0197
  - Weighted-F1 : 0.5755 +/- 0.0189

2. COMPARATIVE RANKING (Macro-F1, Descending):
  1. Extra Trees (NEW)              : 0.5700    <-- THIS EXPERIMENT (NEW)
  2. Random Forest                  : 0.5499  
  3. Linear SVM                     : 0.4720  
  4. RBF SVM                        : 0.4307  
  5. Logistic Regression            : 0.4206  
  6. k-NN (k=5)                     : 0.4102  
  7. WL+SVM kNN-graph h=2           : 0.3911  

3. EMPIRICAL OBSERVATIONS vs. HYPOTHESES:

  Observation A (Performance Relative to Historical Baselines & WL):
    - Extra Trees achieves Macro-F1 = 0.5700 +/- 0.0120, ranking #1 among all evaluated models.
    - It exceeds Random Forest (0.5499) by +0.0201 Macro-F1,
      Linear SVM (0.4720) by +0.0980 Macro-F1,
      Logistic Regression (0.4

In [11]:
# ════════════════════════════════════════════════════════════════════════════
# SECTION 10 - FINAL EXECUTION REPORT & ARTIFACT VERIFICATION
# ════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("FINAL EXECUTION REPORT & WORKSPACE VERIFICATION")
print("=" * 70)
print()

tracked_files = [
    os.path.join(OUT_DIR, "extra_trees_fold_results.csv"),
    os.path.join(OUT_DIR, "extra_trees_summary.csv"),
    os.path.join(OUT_DIR, "extra_trees_config.json"),
    os.path.join(OUT_DIR, "extra_trees_comparison.csv"),
    os.path.join(FIG_DIR, "et_comparison_macro_f1.png"),
    os.path.join(FIG_DIR, "et_comparison_accuracy.png"),
    os.path.join(FIG_DIR, "et_comparison_weighted_f1.png"),
    os.path.join(FIG_DIR, "et_training_time.png"),
    os.path.join(FIG_DIR, "et_prediction_time.png"),
    os.path.join(FIG_DIR, "et_macro_f1_folds.png"),
    os.path.join(FIG_DIR, "et_accuracy_folds.png"),
]

print("Workspace File Verification:")
all_files_exist = True
for p in tracked_files:
    exists = os.path.isfile(p)
    if not exists:
        all_files_exist = False
    size_str = "{:>8,d} bytes".format(os.path.getsize(p)) if exists else "     MISSING"
    status = "[OK]     " if exists else "[MISSING]"
    print("  {} {:32s} : {}".format(status, os.path.basename(p), size_str))

print()
print("Validation Status Summary:")
print("  - Data validation checks : {}/{} checks passed cleanly".format(n_pass, len(val_checks)))
print("  - Folds evaluated        : 5/5 completed successfully")
print("  - All artifacts verified : {}".format("YES" if all_files_exist else "NO"))
print()
print("Status: READY FOR REVIEW")
print()
print("NOTE: Stop after Extra Trees. Do not implement Gradient Boosting, MLP, XGBoost, or LightGBM.")
print("=" * 70)


FINAL EXECUTION REPORT & WORKSPACE VERIFICATION

Workspace File Verification:
  [OK]      extra_trees_fold_results.csv     :    1,165 bytes
  [OK]      extra_trees_summary.csv          :      672 bytes
  [OK]      extra_trees_config.json          :    2,533 bytes
  [OK]      extra_trees_comparison.csv       :    1,251 bytes
  [OK]      et_comparison_macro_f1.png       :   88,691 bytes
  [OK]      et_comparison_accuracy.png       :   86,296 bytes
  [OK]      et_comparison_weighted_f1.png    :   85,951 bytes
  [OK]      et_training_time.png             :   46,200 bytes
  [OK]      et_prediction_time.png           :   60,135 bytes
  [OK]      et_macro_f1_folds.png            :   56,776 bytes
  [OK]      et_accuracy_folds.png            :   60,681 bytes

Validation Status Summary:
  - Data validation checks : 9/9 checks passed cleanly
  - Folds evaluated        : 5/5 completed successfully
  - All artifacts verified : YES

Status: READY FOR REVIEW

NOTE: Stop after Extra Trees. Do not impl